Unified Evaluation Pipeline for CASF (TS-QA + TGQA)

This notebook implements a unified evaluation scaffold for the Contradiction-Aware Sparse Memory Finetuning (CASF) project across two datasets:

- TS-QA (Time-Sensitive QA): evaluates timestamp-conditioned QA with clean vs perturbed (hard-negative) splits.

- TGQA (Temporal Graph QA): converts temporal-graph events into year-conditioned cloze prompts to evaluate time-aware fact completion.

We keep the evaluation abstraction consistent across datasets:

1. Prepare evaluation examples (dataset-specific)

2. Verbalize into prompts (dataset-specific)

3. Model generate function (pluggable)

4. Scoring (exact match / contains / token-F1)

5. ggregate results overall + by slices (source, gap bins, etc.)

This notebook focuses on evaluation structure; model choice can be swapped later (baseline LM, memory-augmented LM, CASF).

**Note: This notebook implements the evaluation infrastructure (dataset normalization → prompt construction → scoring).
Model inference is intentionally left as a stub, since the project baseline will use Llama 3.2 1B Instruct integrated later.

In [1]:
import os, sys
import pandas as pd
import numpy as np

# Ensure repo root on sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.load_data import load_tsqa, load_tgqa

from EvalPipeline.tsqa_eval_utils import (
    to_example as tsqa_to_example,
    split_clean_vs_perturbed,
    evaluate_tsqa,
    bucket_gap_days,
)

from EvalPipeline.tgqa_eval_utils import (
    build_event_list,
    build_cloze_set,
    evaluate_tgqa,
)

2. 

We define a single callable:

- model_generate_fn(prompt: str) -> str

This makes evaluation independent of the model implementation (baseline LM, retrieval-augmented LM, CASF).

In [10]:
def model_generate_fn(prompt: str) -> str:
    raise NotImplementedError(
        "Model not integrated yet. This notebook validates dataset-to-prompt conversion + scoring scaffold. "
        "Plug in Llama 3.2 1B Instruct later by implementing model_generate_fn(prompt)->str."
    )

Note: This notebook implements the evaluation infrastructure (dataset normalization → prompt construction → scoring).
Model inference is intentionally left as a stub, since the project baseline will use Llama 3.2 1B Instruct integrated later.

In [11]:
from EvalPipeline.tsqa_eval_utils import build_prompt

for ex in clean[:2]:
    print(build_prompt(ex)[:600])
    print("GOLD:", ex.answers)
    print("source:", ex.source, "| gap_days:", ex.gap_days, "| hard_neg:", ex.is_hard_negative)
    print("-"*80)

Question: Today is Tuesday, September 24, 2013. Who was the Transport Minister of Thailand?

Context:
Thursday, March 28, 2013. Old school ties key to Australia's role in new order
When Thai Transport Minister Chadchart Sittipunt met a group of Australian journalists on a balmy Bangkok afternoon recently, he was quick to offer a share tip to visitors.
"You should buy stocks in Cochlear," said Dr Sittipunt, a MIT-educated engineering professor who spent 18 months in Australia 10 years ago as a researcher at the CSIRO.
The tip was as much personal as financial.
Sittipunt's son had his hearing re
GOLD: ['Chadchart Sittipunt']
source: streamingqa | gap_days: 180 | hard_neg: False
--------------------------------------------------------------------------------
Question: Today is Friday, March 7, 2014. Where will the first test between Australia and South Africa be held?

Context:
Monday, February 10, 2014. Unless Australia starts scoring when first batting there's unlikely to be any second 

In [12]:
for item in cloze_items[:5]:
    print("PROMPT:", item.prompt)
    print("GOLD:", item.gold, "| year:", item.start_year)
    print("-"*80)

PROMPT: In 1921, Chris Evans was born ____.
GOLD: in San Francisco | year: 1921
--------------------------------------------------------------------------------
PROMPT: In 1977, Chris Evans won ____.
GOLD: prize Oakland Trophy | year: 1977
--------------------------------------------------------------------------------
PROMPT: In 1986, Chris Evans won ____.
GOLD: prize Greenwood Trophy | year: 1986
--------------------------------------------------------------------------------
PROMPT: In 1991, Chris Evans won prize Falcon ____.
GOLD: Award in Biology | year: 1991
--------------------------------------------------------------------------------
PROMPT: In 1993, Chris Evans won prize ____.
GOLD: Golden Star Prize | year: 1993
--------------------------------------------------------------------------------


3. TS-QA Evaluation (QA-style)

We evaluate:

- Clean subset (non-hard-negative)

- Perturbed subset (hard-negative; contradiction stress test)

- Metrics: Exact Match / Contains / Token-F1.

In [4]:
tsqa = load_tsqa()
tsqa_train = tsqa["train"]

# Normalize into TSQAExample objects
tsqa_examples = [tsqa_to_example(ex) for ex in tsqa_train]

clean, pert = split_clean_vs_perturbed(tsqa_examples)

print("TS-QA train:", len(tsqa_examples))
print("Clean:", len(clean))
print("Perturbed:", len(pert))

# Quick peek
print("\nExample (clean):")
print(clean[0])

TS-QA train: 25041
Clean: 20810
Perturbed: 4231

Example (clean):
TSQAExample(ex_id='train-087792', question='Today is Tuesday, September 24, 2013. Who was the Transport Minister of Thailand?', context='Thursday, March 28, 2013. Old school ties key to Australia\'s role in new order\nWhen Thai Transport Minister Chadchart Sittipunt met a group of Australian journalists on a balmy Bangkok afternoon recently, he was quick to offer a share tip to visitors.\n"You should buy stocks in Cochlear," said Dr Sittipunt, a MIT-educated engineering professor who spent 18 months in Australia 10 years ago as a researcher at the CSIRO.\nThe tip was as much personal as financial.\nSittipunt\'s son had his hearing restored by an Australian medical expert with the aid of a Cochlear implant.\nHe is a believer in the Australian technology, which he said had "changed the lives of many people."\nAs Transport Minister, Sittipunt is overseeing one of the largest infrastructure projects in the country\'s history

In [5]:
# Keep small sample initially to validate pipeline
N = 200

res_clean = evaluate_tsqa(model_generate_fn, clean, include_context=True, max_examples=N)
res_pert = evaluate_tsqa(model_generate_fn, pert, include_context=True, max_examples=min(N, len(pert)))

print("TS-QA (clean) results:", res_clean)
print("TS-QA (perturbed) results:", res_pert)

TS-QA (clean) results: {'n': 200.0, 'exact_match': 0.0, 'contains': 0.0, 'f1': 0.0}
TS-QA (perturbed) results: {'n': 200.0, 'exact_match': 0.0, 'contains': 0.0, 'f1': 0.0}


4. TS-QA Slice Analysis (gap buckets + source)

We analyze performance by:

- evidence/question timestamp gap bucket

- dataset source (streamingqa, *_perturbed, etc.)

- This helps diagnose where time drift or contradictions hurt most.

In [6]:
# Make a dataframe for slicing
df_tsqa = pd.DataFrame([{
    "id": ex.ex_id,
    "source": ex.source,
    "gap_bucket": bucket_gap_days(ex.gap_days),
    "is_hard_negative": ex.is_hard_negative,
    "question_type": ex.question_type,
    "has_critical_dimensions": ex.has_critical_dimensions,
} for ex in tsqa_examples])

display(df_tsqa.groupby(["source", "gap_bucket"]).size().to_frame("count").reset_index())
display(df_tsqa.groupby(["source"])["is_hard_negative"].mean().to_frame("hard_negative_rate"))

,source,gap_bucket,count
0,streamingqa,0-7d,2812
1,streamingqa,31-365d,3994
2,streamingqa,365d+,4992
3,streamingqa,8-30d,7980
4,streamingqa_perturbed,0-7d,555
5,streamingqa_perturbed,31-365d,784
6,streamingqa_perturbed,365d+,977
7,streamingqa_perturbed,8-30d,1616
8,timeqa,missing,1032
9,timeqa_perturbed,missing,299


,hard_negative_rate
source,
streamingqa,0.0
streamingqa_perturbed,1.0
timeqa,0.0
timeqa_perturbed,1.0


5. TGQA Evaluation (Cloze-style)

- TGQA does not provide QA pairs directly. We:

- Extract TG events per story

- Parse (fact, start_year) where available

- Convert to year-conditioned cloze prompts

- Evaluate completion accuracy against extracted “object” strings

In [7]:
tgqa = load_tgqa("TGQA_Story_TG_Trans")
tgqa_train = tgqa["train"]

events = build_event_list(tgqa_train)
print("TGQA events:", len(events))

# Use only events with explicit year anchors for time-aware evaluation
cloze_items = build_cloze_set(events, require_year=True)
print("TGQA cloze items (require_year=True):", len(cloze_items))

print("\nSample cloze item:")
print(cloze_items[0])

TGQA events: 4166
TGQA cloze items (require_year=True): 2814

Sample cloze item:
TGQACloze(story_id='story0', prompt='In 1921, Chris Evans was born ____.', gold='in San Francisco', start_year=1921, raw_fact='Chris Evans was born in San Francisco')


In [8]:
N = 200
res_tgqa = evaluate_tgqa(model_generate_fn, cloze_items, max_examples=N)
print("TGQA results:", res_tgqa)

TGQA results: {'n': 200.0, 'exact_match': 0.0, 'contains': 0.0, 'f1': 0.0}


6.  Summary Table

We summarize baseline scores across datasets. This table is the shared interface we will use later to compare:

- baseline LM

- retrieval-augmented LM

- CASF contradiction-aware memory

In [9]:
summary = pd.DataFrame([
    {"dataset": "TS-QA clean", **res_clean},
    {"dataset": "TS-QA perturbed", **res_pert},
    {"dataset": "TGQA cloze", **res_tgqa},
])

display(summary)

,dataset,n,exact_match,contains,f1
0,TS-QA clean,200.0,0.0,0.0,0.0
1,TS-QA perturbed,200.0,0.0,0.0,0.0
2,TGQA cloze,200.0,0.0,0.0,0.0
